In [127]:
import pandas as pd
import numpy as np
import spacy

nlp = spacy.load("en_core_web_sm")

In [128]:
# loading the data
data = pd.read_csv("dataset/mtsamples.csv")
data

,Unnamed: 0,description,medical_specialty,sample_name,transcription,keywords
0,0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller..."
1,1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh..."
2,2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...","bariatrics, laparoscopic gastric bypass, heart..."
3,3,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,"2-D M-MODE: , ,1. Left atrial enlargement wit...","cardiovascular / pulmonary, 2-d m-mode, dopple..."
4,4,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall ...,"cardiovascular / pulmonary, 2-d, doppler, echo..."
...,...,...,...,...,...,...
4994,4994,Patient having severe sinusitis about two to ...,Allergy / Immunology,Chronic Sinusitis,"HISTORY:, I had the pleasure of meeting and e...",NaN
4995,4995,This is a 14-month-old baby boy Caucasian who...,Allergy / Immunology,Kawasaki Disease - Discharge Summary,"ADMITTING DIAGNOSIS: , Kawasaki disease.,DISCH...","allergy / immunology, mucous membranes, conjun..."
4996,4996,A female for a complete physical and follow u...,Allergy / Immunology,Followup on Asthma,"SUBJECTIVE: , This is a 42-year-old white fema...",NaN
4997,4997,Mother states he has been wheezing and coughing.,Allergy / Immunology,Asthma in a 5-year-old,"CHIEF COMPLAINT: , This 5-year-old male presen...",NaN


In [129]:
# PRE PROCessing
import re

ABBREVIATIONS = {
    "pt": "patient",
    "hx": "history",
    "dx": "diagnosis",
    "tx": "treatment",
    "sx": "symptoms",
    "htn": "hypertension",
    "cp": "chest pain",
    "sob": "shortness of breath"
}

def extract_headers(text):
    # extract from raw (uppercase) text before any lowercasing
    headers = re.findall(r'\b([A-Z][A-Z\s]{3,}):', text)
    return " ".join(h.strip().replace(" ", "_") for h in headers)

def expand_abbreviations(text, abbr_dict):
    for abbr, full in abbr_dict.items():
        pattern = r'\b' + re.escape(abbr) + r'\b'
        # append expansion alongside original token — never discard original
        text = re.sub(pattern, lambda m: m.group() + " " + full, text)
    return text

def clean_symbols(text):
    text = re.sub(r'[^a-zA-Z0-9\s.,]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def preprocess(text):
    # used for the spacy NLP pass only (entity/negation extraction)
    text = text.lower()
    text = expand_abbreviations(text, ABBREVIATIONS)
    text = clean_symbols(text)
    return text

In [130]:
GENERIC_TERMS = {
    "patient",
    "history",
    "exam",
    "examination",
    "procedure",
    "doctor",
    "report"
}

def extract_entities(doc):
    entities = []

    for chunk in doc.noun_chunks:
        phrase = chunk.text.lower().strip()

        # skip generic clinical nouns
        if phrase in GENERIC_TERMS:
            continue

        # only keep phrases with a noun root
        if chunk.root.pos_ == "NOUN":
            entities.append({
                "entity": phrase,
                "root": chunk.root,
                "status": "present"
            })

    return entities

In [131]:
NEGATION_CUES = {
    "no",
    "not",
    "denies",
    "without",
    "negative"
}

def detect_negation(entities):
    for entity in entities:
        root = entity["root"]

        # check ancestor verbs (like "denies")
        for ancestor in root.ancestors:
            if ancestor.lemma_.lower() in NEGATION_CUES:
                entity["status"] = "negated"

        # check children for negation tokens
        for child in root.children:
            if child.dep_ == "neg":
                entity["status"] = "negated"

    return entities

In [132]:
def process_document(text):
    headers = extract_headers(text)      # extract from raw before lowercasing
    cleaned = preprocess(text)
    doc = nlp(cleaned)

    entities = extract_entities(doc)
    entities = detect_negation(entities)

    entity_list = [e["entity"] for e in entities]
    negated_list = [e["entity"] for e in entities if e["status"] == "negated"]

    return cleaned, entity_list, negated_list, headers

In [133]:
# data cleaning
data = data.dropna(subset=["transcription"])
data = data[data["transcription"].str.strip() != ""]

data = data.reset_index(drop=True)

print("Remaining rows:", len(data))

# data filter
top_specialties = data['medical_specialty'].value_counts().head(5).index
data = data[data['medical_specialty'].isin(top_specialties)]

print("Remaining rows:", len(data))


Remaining rows: 4966
Remaining rows: 2603


In [134]:
results = data["transcription"].apply(process_document)

In [135]:
data["clean_text"] = results.apply(lambda x: x[0])
data["entities"] = results.apply(lambda x: x[1])
data["negated_entities"] = results.apply(lambda x: x[2])
data["headers"] = results.apply(lambda x: x[3])

In [136]:
def combine_features(row):
    raw = row["transcription"]       # start from raw — never discard signal
    headers = row["headers"]         # boosted header tokens from uppercase text
    negated = " ".join(["NEG_" + e.replace(" ", "_")
                        for e in row["negated_entities"]])
    return raw + " " + headers + " " + negated

In [137]:
data["combined_text"] = data.apply(combine_features, axis=1)

In [138]:
texts = data["combined_text"]
labels = data["medical_specialty"]

In [145]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words=None,       # don't strip clinically meaningful words
    ngram_range=(1, 2),
    sublinear_tf=True,     # log-scale TF, reduces dominance of repeated terms
)

X = vectorizer.fit_transform(texts)

In [146]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    labels,
    test_size=0.2,
    random_state=42,
)

In [147]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000)

model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [148]:
from sklearn.metrics import accuracy_score, classification_report

predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

Accuracy: 0.6065259117082533
                             precision    recall  f1-score   support

 Cardiovascular / Pulmonary       0.23      0.15      0.18        68
 Consult - History and Phy.       0.80      0.90      0.84       109
                 Orthopedic       0.03      0.02      0.02        65
                  Radiology       0.52      0.47      0.50        55
                    Surgery       0.68      0.81      0.74       224

                   accuracy                           0.61       521
                  macro avg       0.45      0.47      0.46       521
               weighted avg       0.55      0.61      0.57       521



In [149]:

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

df = pd.read_csv("dataset/mtsamples.csv").dropna(subset=["transcription", "medical_specialty"])
df["medical_specialty"] = df["medical_specialty"].str.strip()

top5 = df["medical_specialty"].value_counts().head(5).index
df = df[df["medical_specialty"].isin(top5)]

X_raw = df["transcription"]
y_raw = df["medical_specialty"]

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(X_raw)

X_tr, X_te, y_tr, y_te = train_test_split(X_tfidf, y_raw, test_size=0.2, random_state=42)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_tr, y_tr)

preds = clf.predict(X_te)
print(f"Accuracy: {accuracy_score(y_te, preds):.4f}\n")
print(classification_report(y_te, preds))


Accuracy: 0.6334

                            precision    recall  f1-score   support

Cardiovascular / Pulmonary       0.32      0.24      0.27        68
Consult - History and Phy.       0.80      0.87      0.83       109
                Orthopedic       0.11      0.06      0.08        65
                 Radiology       0.55      0.60      0.57        55
                   Surgery       0.71      0.81      0.76       224

                  accuracy                           0.63       521
                 macro avg       0.50      0.52      0.50       521
              weighted avg       0.59      0.63      0.61       521



In [144]:


print("Original Text:\n", data["transcription"][5])
doc = nlp(preprocess(data["transcription"][5]))
print(list(doc.sents))

entities = extract_entities(doc)
entities = detect_negation(entities)
for entity in entities:
    print(entity)

from spacy import displacy
displacy.render(next(doc.sents), style='dep', jupyter=True)



KeyError: 5